# Cross-Domain Deception Detection (DOLOS + RLT)

This notebook contains the full pipeline: data preparation, DANN model, training, validation, and leave-one-domain-out testing.

In [ ]:
# If needed, uncomment and run
# !pip install -r requirements.txt
# !pip install -e .

## 1) Imports and config

In [ ]:
from __future__ import annotations

import json
import random
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, List, Sequence, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from torch.autograd import Function
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
from torchvision.models import ResNet18_Weights, resnet18
from tqdm import tqdm

## 2) Data utilities

In [ ]:
LABEL_MAP = {"truth": 0, "lie": 1}

@dataclass
class DatasetConfig:
    image_size: int = 224
    batch_size: int = 16
    num_workers: int = 2
    train_csv: str = "data/splits/train.csv"
    val_csv: str = "data/splits/val.csv"
    test_csv: str = "data/splits/test.csv"

class DeceptionImageDataset(Dataset):
    def __init__(self, df: pd.DataFrame, transform=None, return_domain: bool=False):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.return_domain = return_domain
        self.domain_to_idx = {d:i for i,d in enumerate(sorted(self.df["domain"].unique().tolist()))}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row["path"]).convert("RGB")
        if self.transform is not None:
            image = self.transform(image)
        y = int(row["label"])
        if self.return_domain:
            d = self.domain_to_idx[row["domain"]]
            return image, torch.tensor(y), torch.tensor(d)
        return image, torch.tensor(y)

def build_transforms(image_size=224):
    train_tfm = transforms.Compose([
        transforms.Resize((image_size+16, image_size+16)),
        transforms.RandomResizedCrop(image_size, scale=(0.7, 1.0)),
        transforms.RandomHorizontalFlip(),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
        transforms.RandomGrayscale(p=0.1),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
    ])
    eval_tfm = transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
    ])
    return train_tfm, eval_tfm

def build_manifest(root: str, output_csv: str, domain_name: str | None = None):
    root_path = Path(root)
    rows = []
    domain = domain_name or root_path.name
    for cls, label in LABEL_MAP.items():
        d = root_path / cls
        if not d.exists():
            continue
        for p in d.glob("**/*"):
            if p.is_file() and p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp"}:
                rows.append({"path": str(p.resolve()), "label": label, "label_name": cls, "domain": domain})
    if not rows:
        raise ValueError(f"No images found under {root}")
    df = pd.DataFrame(rows).sample(frac=1.0, random_state=42).reset_index(drop=True)
    Path(output_csv).parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(output_csv, index=False)
    return df

def prepare_merged_manifest(dolos_root: str, rlt_root: str, output_csv: str, third_root: str | None = None):
    parts = []
    parts.append(build_manifest(dolos_root, "data/manifests/dolos.csv", "dolos"))
    parts.append(build_manifest(rlt_root, "data/manifests/rlt.csv", "rlt"))
    if third_root:
        name = Path(third_root).name.lower().replace(" ", "_")
        parts.append(build_manifest(third_root, f"data/manifests/{name}.csv", name))
    merged = pd.concat(parts, ignore_index=True)
    Path(output_csv).parent.mkdir(parents=True, exist_ok=True)
    merged.to_csv(output_csv, index=False)
    return merged

def make_dataloaders(cfg: DatasetConfig, domain_adversarial=True):
    train_df = pd.read_csv(cfg.train_csv)
    val_df = pd.read_csv(cfg.val_csv)
    test_df = pd.read_csv(cfg.test_csv)
    train_tfm, eval_tfm = build_transforms(cfg.image_size)

    train_ds = DeceptionImageDataset(train_df, train_tfm, return_domain=domain_adversarial)
    val_ds = DeceptionImageDataset(val_df, eval_tfm, return_domain=domain_adversarial)
    test_ds = DeceptionImageDataset(test_df, eval_tfm, return_domain=domain_adversarial)

    counts = train_df["label"].value_counts().to_dict()
    weights = [1.0 / counts[y] for y in train_df["label"].tolist()]
    sampler = WeightedRandomSampler(weights=weights, num_samples=len(weights), replacement=True)

    return {
        "train": DataLoader(train_ds, batch_size=cfg.batch_size, sampler=sampler, num_workers=cfg.num_workers),
        "val": DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False, num_workers=cfg.num_workers),
        "test": DataLoader(test_ds, batch_size=cfg.batch_size, shuffle=False, num_workers=cfg.num_workers),
    }

## 3) DANN model

In [ ]:
class GradReverse(Function):
    @staticmethod
    def forward(ctx, x, lambd):
        ctx.lambd = lambd
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        return grad_output.neg() * ctx.lambd, None

def grad_reverse(x, lambd=1.0):
    return GradReverse.apply(x, lambd)

@dataclass
class ModelConfig:
    pretrained: bool = True
    feature_dim: int = 512
    num_domains: int = 2
    domain_loss_weight: float = 0.2

class DeceptionDANN(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        weights = ResNet18_Weights.IMAGENET1K_V1 if cfg.pretrained else None
        backbone = resnet18(weights=weights)
        self.encoder = nn.Sequential(*list(backbone.children())[:-1])
        self.classifier = nn.Sequential(nn.Linear(cfg.feature_dim, 128), nn.ReLU(), nn.Dropout(0.25), nn.Linear(128, 2))
        self.domain_head = nn.Sequential(nn.Linear(cfg.feature_dim, 128), nn.ReLU(), nn.Dropout(0.25), nn.Linear(128, cfg.num_domains))

    def forward(self, x, lambd=0.0):
        feat = self.encoder(x).flatten(1)
        cls_logits = self.classifier(feat)
        dom_logits = self.domain_head(grad_reverse(feat, lambd))
        return cls_logits, dom_logits

## 4) Train / eval helpers

In [ ]:
def compute_lambda(epoch: int, total_epochs: int):
    p = epoch / max(total_epochs - 1, 1)
    return 2.0 / (1.0 + np.exp(-10 * p)) - 1.0

def evaluate(model, loader, device):
    model.eval()
    ys, ps = [], []
    with torch.no_grad():
        for x, y, _ in loader:
            x = x.to(device)
            y = y.to(device)
            logits, _ = model(x, lambd=0.0)
            pred = logits.argmax(1)
            ys.extend(y.cpu().tolist())
            ps.extend(pred.cpu().tolist())
    return {
        "acc": float(accuracy_score(ys, ps)),
        "f1": float(f1_score(ys, ps, average="macro")),
        "cm": confusion_matrix(ys, ps).tolist(),
        "report": classification_report(ys, ps, output_dict=True),
    }

def train_one_fold(train_csv, val_csv, test_csv, out_dir, num_domains, epochs=20, batch_size=16, image_size=224, lr=1e-4, wd=1e-4, domain_loss_weight=0.2):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    cfg = DatasetConfig(image_size=image_size, batch_size=batch_size, train_csv=train_csv, val_csv=val_csv, test_csv=test_csv)
    loaders = make_dataloaders(cfg, domain_adversarial=True)

    device = "cuda" if torch.cuda.is_available() else "cpu"
    model_cfg = ModelConfig(pretrained=True, num_domains=num_domains, domain_loss_weight=domain_loss_weight)
    model = DeceptionDANN(model_cfg).to(device)

    cls_loss_fn = nn.CrossEntropyLoss()
    dom_loss_fn = nn.CrossEntropyLoss()
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)

    best_f1 = -1.0
    history = []

    for ep in range(epochs):
        model.train()
        lam = compute_lambda(ep, epochs)
        total_loss = 0.0
        for x, y, d in tqdm(loaders["train"], desc=f"Epoch {ep+1}/{epochs}"):
            x, y, d = x.to(device), y.to(device), d.to(device)
            cls_logits, dom_logits = model(x, lambd=lam)
            cls_loss = cls_loss_fn(cls_logits, y)
            dom_loss = dom_loss_fn(dom_logits, d)
            loss = cls_loss + domain_loss_weight * dom_loss
            opt.zero_grad()
            loss.backward()
            opt.step()
            total_loss += loss.item() * x.size(0)
        sched.step()

        val = evaluate(model, loaders["val"], device)
        history.append({"epoch": ep + 1, "train_loss": total_loss / len(loaders["train"].dataset), **{k: val[k] for k in ["acc", "f1"]}})
        if val["f1"] > best_f1:
            best_f1 = val["f1"]
            torch.save(model.state_dict(), out_dir / "best_model.pt")

    model.load_state_dict(torch.load(out_dir / "best_model.pt", map_location=device))
    test = evaluate(model, loaders["test"], device)

    result = {"best_val_f1": best_f1, "test_acc": test["acc"], "test_f1": test["f1"], "test_cm": test["cm"], "history": history}
    (out_dir / "metrics.json").write_text(json.dumps(result, indent=2))
    return result

## 5) Prepare your data (edit paths)

In [ ]:
DOLOS_ROOT = "/path/to/DOLOS_Train"
RLT_ROOT = "/path/to/RLT_Test"
THIRD_ROOT = None  # e.g. "/path/to/third_dataset"

# merged_df = prepare_merged_manifest(DOLOS_ROOT, RLT_ROOT, "data/manifests/all_data.csv", THIRD_ROOT)
# merged_df.head()

## 6) Leave-one-domain-out experiment

In [ ]:
def run_cross_domain(manifest_csv="data/manifests/all_data.csv", out_dir="outputs/cross_domain", epochs=20, batch_size=16):
    df = pd.read_csv(manifest_csv)
    domains = sorted(df["domain"].unique().tolist())
    out_root = Path(out_dir)
    out_root.mkdir(parents=True, exist_ok=True)

    all_results = []
    for holdout in domains:
        fold_dir = out_root / f"holdout_{holdout}"
        split_dir = fold_dir / "splits"
        split_dir.mkdir(parents=True, exist_ok=True)

        test_df = df[df["domain"] == holdout]
        trainval_df = df[df["domain"] != holdout].sample(frac=1.0, random_state=42)

        n_val = max(1, int(0.15 * len(trainval_df)))
        val_df = trainval_df.iloc[:n_val]
        train_df = trainval_df.iloc[n_val:]

        train_csv = split_dir / "train.csv"
        val_csv = split_dir / "val.csv"
        test_csv = split_dir / "test.csv"
        train_df.to_csv(train_csv, index=False)
        val_df.to_csv(val_csv, index=False)
        test_df.to_csv(test_csv, index=False)

        result = train_one_fold(
            train_csv=str(train_csv),
            val_csv=str(val_csv),
            test_csv=str(test_csv),
            out_dir=str(fold_dir / "run"),
            num_domains=max(1, len(domains) - 1),
            epochs=epochs,
            batch_size=batch_size,
        )
        all_results.append({"holdout": holdout, "test_acc": result["test_acc"], "test_f1": result["test_f1"], "best_val_f1": result["best_val_f1"]})

    summary = pd.DataFrame(all_results)
    summary.to_csv(out_root / "summary.csv", index=False)
    return summary

# summary_df = run_cross_domain(manifest_csv="data/manifests/all_data.csv", out_dir="outputs/cross_domain", epochs=20, batch_size=16)
# summary_df

## 7) Notes
- Start with 20 epochs; then tune 30-50 epochs, `domain_loss_weight` in [0.1, 0.5], and image size up to 320 if GPU allows.
- This notebook gives a full reproducible baseline for cross-domain generalization; final accuracy depends on true data distribution.